# StudyPlan (Group 78) — Live Demo

**A local-first study planner and Python API for AI agents.**

Run the cells from top to bottom. The first code cell installs the package from PyPI;
skip it if `studyplan` is already installed on this JupyterHub server.

> If `import studyplan` fails after installing, restart the kernel
> (**Kernel → Restart**) and run the cells again.

In [ ]:
# Run once per JupyterHub server. Skip if studyplan is already installed.
# If the default install hits a permission error, use:
# %pip install --user -q studyplan-group78
%pip install -q studyplan-group78

In [ ]:
from datetime import date, timedelta

import studyplan

TODAY = date.today()


def day(n: int) -> str:
    """Return an ISO date string n days from today."""
    return (TODAY + timedelta(days=n)).isoformat()


print("Today:", TODAY.isoformat())
print("Package:", studyplan.__file__)

## 1. GPA — credit-weighted, not a simple average

In [ ]:
courses = [
    {"name": "Calculus", "credits": 4, "grade": "A"},
    {"name": "Accounting", "credits": 3, "grade": "B+"},
]

gpa = studyplan.calculate_gpa(courses)

print("Courses:", [(c["name"], c["credits"], c["grade"]) for c in courses])
print(f"GPA: {gpa}")
print("Check: (4 x 4.0 + 3 x 3.3) / 7 = 3.70")
print("A 4-credit A counts more than a 3-credit B+.")

## 2. Study plan — deadline-first with a time budget

In [ ]:
tasks = [
    {"name": "Math Homework",  "due_date": day(3),  "est_minutes": 60,  "priority": "Medium"},
    {"name": "CS1302 Project", "due_date": day(7),  "est_minutes": 120, "priority": "High"},
    {"name": "Essay Draft",    "due_date": day(15), "est_minutes": 90,  "priority": "Low"},
]

plan = studyplan.suggest_study_plan(tasks, available_minutes=150)

print(f"{'Task':<18}{'Due date':<14}{'Minutes':>8}")
print("-" * 40)
for item in plan:
    print(f"{item['task_name']:<18}{item['due_date']:<14}{item['allocated_minutes']:>8}")
print("-" * 40)
print(f"{'TOTAL':<18}{'':<14}{sum(i['allocated_minutes'] for i in plan):>8}")
print()
print("Earliest deadline is funded first; the 120-min project gets the")
print("remaining 90 min; the essay gets 0 and is not shown.")

## 3. Deadline urgency and exam countdown

In [ ]:
for offset in (3, 7, 15):
    d = day(offset)
    print(f"{d}: {studyplan.days_left(d):>2} days -> {studyplan.urgency_level(d)}")

exam = day(25)
print(f"Exam on {exam}: {studyplan.exam_countdown(exam)} days remaining")
print("Bands: 0-3 days = High, 4-7 = Medium, 8+ = Low.")

## 4. Quiz generator — structured cards for an agent

In [ ]:
notes = {
    "Demand": "Relationship between price and quantity demanded",
    "Accounts Receivable": "Money owed by customers",
}

quiz = studyplan.create_quiz(notes)
for card in quiz:
    print("Q:", card["question"])
    print("A:", card["answer"])
    print()

## 5. Study timer

In [ ]:
session = studyplan.StudySession()
for minutes in (45, 30, 45):
    session.add_time(minutes)

print("Minutes:", session.total_minutes())
print("Hours:", session.total_hours())

## 6. Run the CLI from inside the notebook (optional)

This uses the current kernel's Python, so it works even if the `studyplan`
executable is not on the JupyterHub `PATH`.

**This writes to `~/.studyplan/data.json` on the JupyterHub server.**
To start from a clean slate, uncomment and run the next cell first.

In [ ]:
# Optional: start from a clean slate for the CLI demo.
# This deletes the local studyplan data file on the JupyterHub server.
# from studyplan.storage import DATA_FILE
# DATA_FILE.unlink(missing_ok=True)
# print("Cleared:", DATA_FILE)

In [ ]:
import os
import subprocess
import sys


def run_cli(*args):
    result = subprocess.run(
        [sys.executable, "-m", "studyplan.cli", *args],
        capture_output=True,
        encoding="utf-8",
        errors="replace",
        env={**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUTF8": "1"},
    )
    print("$ studyplan", " ".join(args))
    print(result.stdout.strip())
    if result.returncode != 0:
        print(result.stderr.strip())
    print()


run_cli("init", day(40))
run_cli("add", "Math Homework", "--due", day(3), "--est", "60", "--prio", "Medium")
run_cli("add", "CS1302 Project", "--due", day(7), "--est", "120", "--prio", "High")
run_cli("add", "Essay Draft", "--due", day(15), "--est", "90", "--prio", "Low")
run_cli("plan", "--mins", "150")
run_cli("add-course", "Calculus", "--credits", "4", "--grade", "A")
run_cli("add-course", "Accounting", "--credits", "3", "--grade", "B+")
run_cli("calc-gpa")

## 7. Local-first storage

Everything the CLI just did lives in one JSON file in **your server home**.
No network calls, no account, no telemetry.

In [ ]:
from studyplan.storage import DATA_FILE
import json

print("Data file:", DATA_FILE)

if DATA_FILE.exists():
    data = json.loads(DATA_FILE.read_text(encoding="utf-8"))
    print("Assignments:", len(data.get("assignments", [])))
    print("Courses:", len(data.get("courses", [])))
    print("Study minutes:", data.get("study_minutes", 0))
else:
    print("No data file yet. Run the CLI cell in section 6 first.")

## 8. What to say

- **Ten CLI commands, seven API functions, one dependency, one local JSON file.**
- The planner is **deadline-first**; priority only breaks ties on the same date.
- GPA is **credit-weighted**.
- Data never leaves the machine, so an agent can run fully locally.